In [1]:
## Frameed window

import pygame
import win32gui
import win32con

# Initialize Pygame and font
pygame.init()
pygame.font.init()
clock = pygame.time.Clock()

# Load NES background image and get size
background = pygame.image.load("nes.png")
image_width, image_height = background.get_size()

# Create resizable window with frame, initially sized to image
screen = pygame.display.set_mode((image_width, image_height), pygame.RESIZABLE)
pygame.display.set_caption("NES Controller Window")

# Get window handle
hwnd = pygame.display.get_wm_info()["window"]

# Wait a moment to let window size stabilize
pygame.display.flip()
pygame.time.wait(100)

# Measure the full window (including borders and title bar)
left, top, right, bottom = win32gui.GetWindowRect(hwnd)
window_width = right - left
window_height = bottom - top

# Get the actual client area size (drawable surface)
client_rect = win32gui.GetClientRect(hwnd)
client_width = client_rect[2]
client_height = client_rect[3]

# Calculate border sizes
border_width = (window_width - client_width)
title_height = (window_height - client_height)

# Get screen dimensions
display_info = pygame.display.Info()
screen_width = display_info.current_w

# Calculate position to place window in upper-right corner
x_pos = screen_width - window_width
y_pos = 0

# Move and pin window to top-right
win32gui.SetWindowPos(
    hwnd,                    # Handle to our window
    win32con.HWND_TOPMOST,   # Place it above all non-topmost windows
    x_pos,                   # x position
    y_pos,                   # y position
    window_width,            # Width
    window_height,           # Height
    win32con.SWP_SHOWWINDOW  # Flag to ensure the window is shown and activated correctly.
)

# --------- TextPrint class for displaying text ---------
class TextPrint:
    def __init__(self):
        self.reset()
        self.font = pygame.font.Font(None, 25)

    def tprint(self, screen, text):
        text_bitmap = self.font.render(text, True, (0, 0, 0))
        screen.blit(text_bitmap, (self.x, self.y))
        self.y += self.line_height

    def reset(self):
        self.x = 10
        self.y = 10
        self.line_height = 15

    def indent(self):
        self.x += 10

    def unindent(self):
        self.x -= 10

# Initialize text printer and joysticks
text_print = TextPrint()
joysticks = {}

pygame.joystick.init()

# Detect connected joysticks
for i in range(pygame.joystick.get_count()):
    joy = pygame.joystick.Joystick(i)
    joy.init()
    joysticks[joy.get_instance_id()] = joy
    print(f"Joystick {joy.get_instance_id()} connected: {joy.get_name()}")

print("Setup complete. Run the next cell for the main loop.")

## This is for the simple NES controller
done = False

while not done:
    for event in pygame.event.get():
        if event.type == pygame.QUIT:
            done = True

        elif event.type == pygame.JOYDEVICEADDED:
            joy = pygame.joystick.Joystick(event.device_index)
            joy.init()
            joysticks[joy.get_instance_id()] = joy
            print(f"Joystick {joy.get_instance_id()} connected: {joy.get_name()}")

        elif event.type == pygame.JOYDEVICEREMOVED:
            if event.instance_id in joysticks:
                print(f"Joystick {event.instance_id} disconnected")
                del joysticks[event.instance_id]

        elif event.type == pygame.JOYAXISMOTION:
            print(f"Joystick {event.instance_id} axis {event.axis} moved to {event.value}") # {event.value:.20f}")

    # Clear with white for transparency
    screen.fill((255, 255, 255))
    # Draw NES background
    screen.blit(background, (0, 0))
    # Reset text print position
    text_print.reset()

    # Display mouse coordinates
    mouse_x, mouse_y = pygame.mouse.get_pos()
    text_print.tprint(screen, f"Mouse: ({mouse_x}, {mouse_y})")

    # Display joystick info
    joystick_count = pygame.joystick.get_count()
    text_print.tprint(screen, f"Number of joysticks: {joystick_count}")
    text_print.indent()

    for joystick in joysticks.values():
        jid = joystick.get_instance_id()
        text_print.tprint(screen, f"Joystick {jid}")
        text_print.indent()

        text_print.tprint(screen, f"Name: {joystick.get_name()}")
        text_print.tprint(screen, f"GUID: {joystick.get_guid()}")
        text_print.tprint(screen, f"Power level: {joystick.get_power_level()}")

        axes = joystick.get_numaxes()
        text_print.tprint(screen, f"Number of axes: {axes}")
        text_print.indent()
        for i in range(axes):
            axis_val = joystick.get_axis(i)
            text_print.tprint(screen, f"Axis {i}: {axis_val:.3f}")
        text_print.unindent()

        buttons = joystick.get_numbuttons()
        text_print.tprint(screen, f"Number of buttons: {buttons}")
        text_print.indent()
        for i in range(buttons):
            button_val = joystick.get_button(i)
            text_print.tprint(screen, f"Button {i}: {button_val}")
        text_print.unindent()

        hats = joystick.get_numhats()
        text_print.tprint(screen, f"Number of hats: {hats}")
        text_print.indent()
        for i in range(hats):
            hat_val = joystick.get_hat(i)
            text_print.tprint(screen, f"Hat {i}: {hat_val}")
        text_print.unindent()

        text_print.unindent()
    
        axis_0_val = joystick.get_axis(0)  # Typically left stick X
        # axis_1_val = joystick.get_axis(1)  # Typically left stick Y
        axis_4_val = joystick.get_axis(4)  # Typically right stick X? (Check your printout to be sure)


        # Define a threshold. A value beyond this is considered "pressed".
        threshold = 0.9

        if joystick.get_button(1) == 1:
            pygame.draw.circle(screen, (255, 0, 0), (590, 210), 30)  # Draw red circle at (100, 100)
        if joystick.get_button(2) == 1:
            pygame.draw.circle(screen, (255, 0, 0), (498, 210), 30)  # Draw red circle at (100, 100)
        if joystick.get_button(8) == 1:
            pygame.draw.circle(screen, (255, 0, 0), (285, 208), 30)  # Draw red circle at (100, 100)
        if joystick.get_button(9) == 1:
            pygame.draw.circle(screen, (255, 0, 0), (375, 208), 30)  # Draw red circle at (100, 100)            

        if axis_0_val <= -threshold:
            pygame.draw.circle(screen, (255, 0, 0), (85, 175), 30)  # Left
            #TODO: turn left here
        if axis_0_val >= threshold:
            pygame.draw.circle(screen, (255, 0, 0), (165, 175), 30)  
            # pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)  # Right
            #TODO: Turn right here
        if axis_4_val <= -threshold:
            pygame.draw.circle(screen, (255, 0, 0), (128, 130), 30)  # Up
            # TODO: control both motors in positive direction 255pwm here
        if axis_4_val >= threshold:
            pygame.draw.circle(screen, (255, 0, 0), (128, 216), 30)  # Down
            # TODO: control both motors in negative direction 255pwm here


        # Check Right Stick or Triggers (Axis 4 and 5)
        # Based on your printout, axis 4 goes to 1.0 and -1.0. Let's map it.
        # if axis_4_val <= -threshold:
        #     pygame.draw.circle(screen, (0, 0, 255), (500, 174), 30)  # e.g., Right Stick Left (using blue to differentiate)
        # if axis_4_val >= threshold:
        #     pygame.draw.circle(screen, (0, 0, 255), (548, 216), 30)  # e.g., Right Stick Right
    text_print.unindent()

    pygame.display.flip()
    clock.tick(30)

pygame.quit()

C:\Users\micha\AppData\Roaming\Python\Python313\site-packages\pygame\pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists


pygame 2.6.1 (SDL 2.28.4, Python 3.13.9)
Hello from the pygame community. https://www.pygame.org/contribute.html
Joystick 0 connected: USB Gamepad
Setup complete. Run the next cell for the main loop.
Joystick 0 connected: USB Gamepad
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to -1.000030518509476
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to -1.000030518509476
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to 1.0
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to -1.000030518509476
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to -1.000030518509476
Joystick 0 axis 4 moved to -0.007812738425855281
Joystick 0 axis 4 moved to 1.0
Joystick 0 axis 4 moved to -0.007812738425855281
